<a href="https://colab.research.google.com/github/yuki-gu/music-colab-notebooks/blob/main/MIDI%E6%95%B4%E5%BD%A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import files
uploaded = files.upload()

if len(uploaded) == 0:
  raise Exception('No file uploaded.')

input_files = list(uploaded.keys())
print(input_files)

Saving test_file1.mid to test_file1.mid
Saving test_file2.mid to test_file2.mid
['test_file1.mid', 'test_file2.mid']


In [1]:
!pip install mido pretty_midi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 26.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.0 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=1513a310f0ed792b7a7e6281fc8ac58b99c8ebd59a6392bf4c5564dc11c1b479
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [2]:
#@title MIDIの整形
delete_end_notes = False #@param {type:"boolean"}
merge_voice = True #@param {type:"boolean"}
delete_high_notes = False #@param {type:"boolean"}
delete_ties = True #@param {type:"boolean"}
extend_short_notes = True #@param {type:"boolean"}

import sys
import pretty_midi as pm
import mido

def normalize_midi(input_file, output_file):
  mid = mido.MidiFile(input_file)

  # 末尾の音符の削除
  if delete_end_notes:
    for t in mid.tracks[1:]:
      del t[-5:-1]

  # 声部をマージ
  if merge_voice:
    for i, t in enumerate(mid.tracks[1:]):
      for j, msg in enumerate(t):
        if not msg.is_meta:
          t[j] = msg.copy(channel=i)

  # 高音を削除
  if delete_high_notes:
    track = mid.tracks[1]
    bar_duration = mid.ticks_per_beat * 4  # 1小節のティック数
    delete_index = []

    max_pitch = -1
    delete_off_notes = []
    timer = 0
    for i, msg in enumerate(track):
      timer += msg.time
      if msg.type == 'note_on':
        # 前のonノートに対応するoffノートは存在しない
        if msg.note in delete_off_notes:
          delete_off_notes.remove(msg.note)

        if 0 <= max_pitch and 77 < msg.note and max_pitch + 12 < msg.note:
          delete_index.append(i)
          delete_off_notes.append(msg.note)
        elif max_pitch < msg.note:
          max_pitch = msg.note  # 上限を超えたらすぐに上限を調整
          timer = 0  # 一定時間上限が下がらないように維持
        elif timer > bar_duration:
          max_pitch = msg.note  # 上限を下げる
          timer = 0  # 一定時間維持
      if msg.type == 'note_off':
        if msg.note in delete_off_notes:
          delete_index.append(i)  # オフノートを削除
          delete_off_notes.remove(msg.note)

    for i in reversed(delete_index):
      if i+1 <= len(track)-1:
        # 削除するノートのtimeを後ろのノートに加算
        track[i+1] = track[i+1].copy(time=track[i+1].time + track[i].time)
      del track[i]

  # タイを削除
  if delete_ties:
    CHORD_TIME_THRESHOLD = 40
    # 前提: noteのtime = 前のnoteから発音までに待機する時間
    # 要件:
    # ・(note_on: time<CHORD_TIME_THRESHOLD)は同時発声 -> そのまま残す
    # ・(note_on: time>0)の時は前音が残った状態で発声 -> 前音をnote_off
    # 処理内容:
    # ・on状態のnoteを監視(リスト管理) -> 次の(note_on: time>0)が来たらnote_offを追加
    # ・(note_off: time>0)(note_on: time=0)でも処理を行う必要 -> 前のnote_onから時間が経過しているかをbool変数で管理
    # ・各trackに対して実行する。追加するnoteは msg.copy() によって作成する。
    # 既に存在しているnote_off: 現段階では放置(影響なし)
    # time=0で 新音のON -> 前音のOFF の順番だと前音のOFFが追加され2つになる: 現段階では放置(影響なし)
    for i, t in enumerate(mid.tracks[1:]):
      pressed_notes = []  # note_onになっている音のリスト(Mido message のリスト)
      is_notes_in_chord = False  # 同時押しの音符かどうか
      insert_messages = []
      for j, msg in enumerate(t):
        if msg.time > CHORD_TIME_THRESHOLD:
          is_notes_in_chord = False
        if msg.type == 'note_on':
          if not is_notes_in_chord and pressed_notes:  # 既に押されている & 同時押しでない
            # pressed_notes内の音をnote_off (今のmsgの直前に挿入)
            # forのためtrackの変更は後回し
            insert_messages.append((
                j,
                [mido.Message('note_off', channel=m.channel, note=m.note, velocity=m.velocity, time=0)
                    for m in pressed_notes]
            ))
            pressed_notes.clear()
          pressed_notes.append(msg)
          is_notes_in_chord = True
        if msg.type == 'note_off':
          # pressed_notes内もoffに
          pressed_notes = [m for m in pressed_notes if m.note != msg.note]
      # trackの内容を変更
      for i, msg_lst in reversed(insert_messages):
        for msg in reversed(msg_lst):
          # insert: Insert object before index
          t.insert(i+1, msg)  # t[i].time の待機後 -> t[i] の後に挿入

  mid.save(output_file)


  m = pm.PrettyMIDI(output_file)
  if extend_short_notes:
    tempo = m.get_tempo_changes()[1][0]
    min_sec = 60 / tempo / 4  # 16分音符
    for inst in m.instruments:
        for n in inst.notes:
            if n.end - n.start < min_sec:
                n.end = n.start + min_sec
  m.write(output_file)

In [8]:
# @title 整形の実行
import os
from google.colab import files

output_files = []
for midi_file in input_files:
  base, ext = os.path.splitext(midi_file)
  output_file = base + "_fmt" + ext
  normalize_midi(midi_file, output_file)
  output_files.append(output_file)

if len(output_files) == 1:
  files.download(output_file)
else:
  !zip output.zip {" ".join(output_files)}
  files.download("output.zip")

updating: test_file1_fmt.mid (deflated 37%)
updating: test_file2_fmt.mid (deflated 37%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>